In [3]:
"""Train the KNN detector and scale its anomaly scores."""

import os
import sys

# Clear any cached modules
if 'config' in sys.modules:
    del sys.modules['config']
if 'to_tflite' in sys.modules:
    del sys.modules['to_tflite']

import pandas as pd
from pyod.models.knn import KNN
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import config
import to_tflite

# Build the detector from its configuration.
knn = KNN(
    contamination=config.CONTAMINATION,
    n_neighbors=config.N_NEIGHBORS,
    method=config.METHOD,
)

# NOTE: all dataset features must be pre-normalized so that, after the int8
# quantization in keras/to_tflite.py, no value overflows the int8 range.
# Fit the configured features from the CSV.
features = pd.read_csv(config.CSV_PATH)[config.FEATURES].to_numpy()
X_train, X_test = train_test_split(
    features, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)
knn.fit(X_train)

# Scale the raw anomaly scores into the configured range.
raw_scores = knn.decision_function(X_train).reshape(-1, 1)
score_scaler = MinMaxScaler(feature_range=config.SCORE_RANGE).fit(raw_scores)
train_scores = score_scaler.transform(raw_scores).ravel()
test_scores = score_scaler.transform(
    knn.decision_function(X_test).reshape(-1, 1)
).ravel()

# Learn the scores with a small network and export .keras + .tflite.
to_tflite.to_tflite(
    name="knn",
    x_train=X_train,
    y_train=train_scores,
    x_test=X_test,
    y_test=test_scores,
    layers=config.LAYERS,
    learning_rate=config.LEARNING_RATE,
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    verbose=config.VERBOSE,
    feature_names=config.FEATURES,
)


2026-08-14 18:27:19.469409: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 1/50


2026-08-14 18:27:32.078465: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


3747/3747 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.0011 - val_loss: 8.8339e-04
Epoch 2/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 6.9689e-04 - val_loss: 7.8157e-04
Epoch 3/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 6.8807e-04 - val_loss: 8.7143e-04
Epoch 4/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 6.7669e-04 - val_loss: 7.6732e-04
Epoch 5/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 6.7158e-04 - val_loss: 7.5828e-04
Epoch 6/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 6.6644e-04 - val_loss: 7.9371e-04
Epoch 7/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 6.6586e-04 - val_loss: 7.5775e-04
Epoch 8/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 6.6191e-04 - val_loss: 7.5154e-04
Epoch 9/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - loss: 6.5797e-04 - val_loss: 7.5901e-04
Epoch 10/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - loss: 6.5645e-04 - val_loss: 8.4551e-04
Epoch 11/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 912us/st

INFO:tensorflow:Assets written to: /tmp/tmpt00d0ioo/assets


Saved artifact at '/tmp/tmpt00d0ioo'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139864388339664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139864388343312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139864388341968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139864388340624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139864388343888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139864388340816: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/patrik/Diplomka/LODA/env/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1786725166.944161   21219 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786725166.944179   21219 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-08-14 18:32:46.944399: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpt00d0ioo
2026-08-14 18:32:46.944729: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-08-14 18:32:46.944734: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpt00d0ioo
I0000 00:00:1786725166.954269   21219 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-08-14 18:32:46.954605: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-08-14 1

<Sequential name=sequential, built=True>